# Lab 6.5 &mdash; Challenge: Score the Two Halves Apart

**Level:** Advanced &middot; challenge &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 2 &middot; Module 6 &mdash; Agentic RAG**

### What you'll do
- Label an eval set: which section <em>should</em> have come back, for each question
- Measure retrieval with recall@k and precision@k, and choose k with the numbers
- Measure generation with faithfulness and answer relevance &mdash; no labels needed
- Put a run in the 2&times;2 and read off which half to fix

> **How this lab works.** You write real LangChain code. Fill every `BLANK`, then run the
> **Self-check** cell under each section &mdash; those check the *objects you built* (a chunked
> `Document`, a Chroma collection, a bound tool, a compiled graph, a parser), so they are
> deterministic and do not depend on the model. Cells marked **Run it for real** put your code in
> front of the sandbox model; that is the part worth watching. The score line is feedback, not a
> grade.

> **The bridge into Day 3.** One end-to-end score cannot tell a lucky answer from a
> grounded one. Two scores can, and the fixes are unrelated.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-6-05")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens. Thinking is off by default here because you will make a lot of calls today;
# pass think=True to any call below to see the difference for yourself.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------- the corpus (synthetic, self-contained)
# Two short operating documents about the same payments. Read 3.2: the rule and the exception
# that qualifies it are adjacent sentences, which is the whole of Lab 6.1's first lesson. Note
# also what is NOT here -- nothing mentions FX or hedging anywhere, and Lab 6.4 needs that gap.

DOCS = {
    "ops-runbook-v4.md": """## 3.1 Insufficient funds
A payment returned INSUFFICIENT_FUNDS is retried once after 24 hours. If the retry also fails,
notify the client desk. Operations must not fund the account manually.

## 3.2 Limit breaches
Payments above USD 500,000 require Treasury approval before release. This does not apply to
intra-group transfers, which settle same-day without any approval.

## 3.3 Invalid beneficiary details
A payment returned INVALID_IBAN is returned to the originator with code R04. Beneficiary
details are never repaired in-house.

## 3.4 Sanctions review
A payment held for SANCTIONS_REVIEW is decided by Compliance. Operations must not release or
cancel it under any circumstances.
""",
    "escalation-policy-v2.md": """## 1 Approval authority
A duty manager may approve a release up to USD 250,000. Above that figure Treasury approval is
required, and must be recorded against the payment reference.

## 2 Escalation timers
If an approver has not responded within 15 minutes, escalate to the Treasury lead, and after a
further 15 minutes to the head of operations.
""",
}

print(f"{len(DOCS)} documents, {sum(len(d) for d in DOCS.values())} characters")

In [ ]:
# ------------------------------------------------- the embedding model (nothing to fill in)
# The sandbox has no egress, and chromadb's DEFAULT embedding function downloads about 80 MB
# of ONNX model the first time it is called. So this module brings its own: one hashed bucket
# per meaningful word, normalised to unit length. It is arithmetic rather than learning, which
# is the point -- it runs offline, it is deterministic, and you can read every line of it.
#
# What it CAN do: score two texts by the words they share. What it CANNOT do: match meaning
# with no words in common. Lab 6.2 is about living with exactly that.
import re, math, hashlib
from langchain_core.embeddings import Embeddings

STOP = set("""a an the of for is are was were do does did what which who this that these those it
its to in on at by with from about and or not no be been have has had can could should would will
you your we our i me my how why when where there here as if then than so such only just also very
more most some any other""".split())

def content_words(text: str) -> list:
    """The words worth indexing: lower-cased, no punctuation, no stop words."""
    return [w for w in re.findall(r"[a-z0-9_]+", (text or "").lower())
            if w not in STOP and len(w) > 1]


class LabEmbeddings(Embeddings):
    """A tiny embedding model you can read. Same interface as any other LangChain embedding."""

    dim = 1024                      # enough buckets that two different words rarely collide

    def _vector(self, text: str) -> list:
        vec = [0.0] * self.dim
        for word in content_words(text):
            bucket = int(hashlib.sha256(word.encode()).hexdigest()[:8], 16) % self.dim
            vec[bucket] += 1.0
        length = math.sqrt(sum(x * x for x in vec)) or 1.0
        return [x / length for x in vec]      # unit length, so cosine is just a dot product

    def embed_documents(self, texts: list) -> list:
        return [self._vector(t) for t in texts]

    def embed_query(self, text: str) -> list:
        return self._vector(text)


print("embeddings:", LabEmbeddings.dim, "dimensions, offline, deterministic")

In [ ]:
# ------------------------------------------------- carried forward from Lab 6.1 (nothing to fill in)
# Exactly what you built in Lab 6.1: split on headings, index in Chroma, search with a floor.
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_chroma import Chroma

FLOOR = 0.20            # the similarity a chunk must clear to be used at all (Lab 6.1)

def section_chunks() -> list:
    """One Document per '##' section, with the heading kept in the text and in the metadata."""
    splitter = MarkdownHeaderTextSplitter(headers_to_split_on=[("##", "section")],
                                          strip_headers=False)
    out = []
    for name, text in DOCS.items():
        for chunk in splitter.split_text(text):
            chunk.metadata["source"] = name
            out.append(chunk)
    return out


_store = None
def store():
    """The Chroma collection, built once, on first use."""
    global _store
    if _store is None:
        chunks = section_chunks()
        _store = Chroma(collection_name="module6-corpus",
                        embedding_function=LabEmbeddings(),
                        persist_directory=os.path.join(WORK, "chroma"),
                        collection_configuration={"hnsw": {"space": "cosine"}})
        # ids derived from the chunk, so re-running this notebook updates instead of duplicating
        _store.add_documents(chunks, ids=[f"{c.metadata['source']}#{c.metadata['section']}"
                                          for c in chunks])
    return _store


def search(query: str, k: int = 4, floor: float = 0.0, where: dict | None = None) -> list:
    """Top-k from the store as plain dicts, with anything below `floor` dropped."""
    hits = store().similarity_search_with_score(query, k=k, filter=where)
    out = []
    for doc, distance in hits:
        similarity = 1.0 - distance         # cosine space: 1.0 identical, 0.0 nothing in common
        if similarity >= floor:
            out.append({"score": round(similarity, 3), "text": doc.page_content,
                        "source": doc.metadata["source"], "section": doc.metadata["section"]})
    return out


print(f"index ready: {len(store().get()['ids'])} chunks")

## Concept

A RAG system is two systems. They fail differently, they are fixed differently, and a single
end-to-end score cannot tell you which one broke.

The worst box in the 2&times;2 is **right answer, wrong evidence** &mdash; the model knew it already, or
guessed well. It passes every demo, and it stops working the day the model changes.

## Section 1 &mdash; Label the eval set, then choose k

Anything measured `@k` needs a ground truth: for this question, which section *should* come back?
Producing those labels is the actual work, and there is no shortcut.

Then k. Raising it always helps recall and always hurts precision, so &ldquo;pick a good k&rdquo; is not a
judgement call &mdash; it is a rule you state and then read off the sweep.

In [ ]:
# (question, the section that answers it, one true claim quoted from that section)
LABELLED = [
    ("What approval is needed above USD 500,000?", "3.2 Limit breaches",
     "Payments above USD 500,000 require Treasury approval before release"),
    ("What happens to a payment returned INVALID_IBAN?", "3.3 Invalid beneficiary details",
     "A payment returned INVALID_IBAN is returned to the originator with code R04"),
    ("Who decides on a payment held for sanctions review?", "3.4 Sanctions review",
     "A payment held for SANCTIONS_REVIEW is decided by Compliance"),
    ("How much may a duty manager approve?", "1 Approval authority",
     "A duty manager may approve a release up to USD 250,000"),
    ("What happens when a payment is returned INSUFFICIENT_FUNDS?", "3.1 Insufficient funds",
     "A payment returned INSUFFICIENT_FUNDS is retried once after 24 hours"),
    ("How long before an unanswered approval escalates?", "2 Escalation timers",
     "If an approver has not responded within 15 minutes, escalate to the Treasury lead"),
]

def retrieved_sections(question: str, k: int) -> list:
    return [r["section"] for r in search(question, k=k)]


def recall_at_k(k: int) -> float:
    """Fraction of questions whose answering section came back at all."""
    return sum(1 for q, want, _ in LABELLED
               if want in retrieved_sections(q, k)) / len(LABELLED)


def precision_at_k(k: int) -> float:
    """Of everything returned across the eval set, what fraction was the right section?"""
    returned = sum(len(retrieved_sections(q, k)) for q, _, _ in LABELLED)
    correct = sum(1 for q, want, _ in LABELLED if want in retrieved_sections(q, k))
    return correct / returned if returned else 0.0

In [ ]:
# The sweep. Read it before you choose anything.
def _sweep():
    for k in (1, 2, 3, 4, 5):
        print(f"  k={k}   recall {recall_at_k(k):.0%}   precision {precision_at_k(k):.0%}")
guard(_sweep)

In [ ]:
def chosen_k() -> int:
    """How many chunks to retrieve.

    The rule, stated before looking: the SMALLEST k that finds the answering section for
    every question in the eval set. Anything larger only adds noise; anything smaller
    loses an answer.
    """
    return 3          # recall is 100% from k=3 up, and precision only falls after that

In [ ]:
# --- Self-check: Section 1   (labels and metrics over the real index -- no model)
check("every labelled section really exists in the index",
      lambda: all(want in [m["section"] for m in store().get()["metadatas"]]
                  for _, want, _ in LABELLED))
check("every labelled claim really appears in its section",
      lambda: all(" ".join(claim.split()).lower() in
                  " ".join(next(d for d, m in zip(store().get()["documents"],
                                                  store().get()["metadatas"])
                                if m["section"] == want).split()).lower()
                  for _, want, claim in LABELLED),
      "a mislabelled ground truth measures your labelling, not your retriever")
check("recall never decreases as k grows",
      lambda: recall_at_k(5) >= recall_at_k(3) >= recall_at_k(1))
check("precision does the opposite -- more results, more noise",
      lambda: precision_at_k(1) > precision_at_k(5),
      "that trade is the whole of retrieval tuning")
check("at k=1 precision and recall are the same number",
      lambda: abs(precision_at_k(1) - recall_at_k(1)) < 1e-9)
check("your k finds every answering section",
      lambda: recall_at_k(chosen_k()) == 1.0)
check("and it is the smallest k that does",
      lambda: recall_at_k(chosen_k() - 1) < 1.0,
      "a larger k passes the check above too, and pays for chunks nothing needed")

## Section 2 &mdash; The half you can measure without labels

Faithfulness and answer relevance need only what is already in a trace: the question, the
retrieved text, and the answer. No ground truth, so this is the half you can start measuring on
Monday.

In [ ]:
def normalise(t: str) -> str:
    return " ".join((t or "").split()).lower()


def faithfulness(claims: list, results: list) -> float:
    """Fraction of the answer's claims that the retrieved text actually contains."""
    if not claims:
        return 0.0      # saying nothing is not the same as saying only supported things
    context = normalise(" ".join(r["text"] for r in results))
    return sum(1 for c in claims if normalise(c) in context) / len(claims)


def answer_relevance(question: str, claims: list) -> float:
    """How much of what the question asked about the answer actually addresses."""
    asked = set(content_words(question))
    if not asked:
        return 0.0
    answered = set(content_words(" ".join(claims)))
    return len(asked & answered) / len(asked)

In [ ]:
# --- Self-check: Section 2   (metrics on fixed strings and real retrievals -- no model)
Q0, WANT0, TRUE0 = LABELLED[0]
HITS0 = search(Q0, k=3)
LIE0 = "Payments above USD 500,000 may be released by the duty manager"

check("a fully supported answer is perfectly faithful",
      lambda: faithfulness([TRUE0], HITS0) == 1.0)
check("an invented claim is not",
      lambda: faithfulness([LIE0], HITS0) == 0.0)
check("a half-invented answer scores half",
      lambda: abs(faithfulness([TRUE0, LIE0], HITS0) - 0.5) < 1e-9,
      "faithfulness is per claim, which is what makes it actionable")
check("an empty answer is not faithful by default",
      lambda: faithfulness([], HITS0) == 0.0,
      "score it 1.0 and every refusal becomes your best-performing answer")
check("an on-topic answer is relevant",
      lambda: answer_relevance(Q0, [TRUE0]) > 0.5)
check("a perfectly grounded answer to a DIFFERENT question is faithful and irrelevant",
      lambda: faithfulness([LABELLED[2][2]], search(LABELLED[2][0], k=3)) == 1.0
              and answer_relevance(Q0, [LABELLED[2][2]]) < 0.4,
      "the two metrics are independent, which is exactly why you need both")
check("neither metric needed a label",
      lambda: faithfulness([TRUE0], HITS0) == 1.0 and answer_relevance(Q0, [TRUE0]) > 0.0)

## Section 3 &mdash; Read the 2&times;2

Two questions per run: *did the right evidence come back?* and *is the answer supported by what
did?* Four combinations, and each one names a different half to fix.

In [ ]:
def boxes() -> dict:
    """(right evidence?, faithful?) -> (what happened, which half to fix)."""
    return {
        (True,  True):  ("grounded",                   "nothing"),
        (True,  False): ("ignored the evidence",       "generation"),
        (False, True):  ("faithful to the wrong text", "retrieval"),
        (False, False): ("unsupported",                "both"),
    }


def diagnose(question: str, want_section: str, claims: list, k: int = 3) -> dict:
    """Which of the four boxes is this run in, and what should be fixed?"""
    results = search(question, k=k)
    got_evidence = want_section in [r["section"] for r in results]
    faithful = faithfulness(claims, results) == 1.0
    box, fix = boxes()[(got_evidence, faithful)]
    return {"box": box, "fix": fix, "evidence": got_evidence, "faithful": faithful}

In [ ]:
# --- Self-check: Section 3   (the diagnosis, on real retrievals -- no model)
def missed_section(question, k=3):
    """A real section this question does NOT retrieve -- computed, not assumed.

    Hard-coding one is how you write a check that passes for the wrong reason: the section
    you picked as 'wrong' may well be in the top k.
    """
    got = retrieved_sections(question, k)
    return next(m["section"] for m in store().get()["metadatas"] if m["section"] not in got)

def quoted_from_top(question, k=3):
    """A claim lifted verbatim from whatever DID come back, so it is faithful by construction."""
    return search(question, k=k)[0]["text"][:60]

check("right evidence, supported claim -> grounded, nothing to fix",
      lambda: diagnose(Q0, WANT0, [TRUE0])["fix"] == "nothing")
check("right evidence, invented claim -> fix generation",
      lambda: diagnose(Q0, WANT0, [LIE0])["fix"] == "generation",
      "the evidence was sitting right there and the answer went past it")
check("the counterfactual section really is one that did not come back",
      lambda: missed_section(Q0) not in retrieved_sections(Q0, 3))
check("wrong evidence, and a claim supported by whatever DID come back -> fix retrieval",
      lambda: diagnose(Q0, missed_section(Q0), [quoted_from_top(Q0)])["fix"] == "retrieval",
      "faithful to the text in front of it, and the text in front of it was the wrong text")
check("wrong evidence and an invented claim -> fix both",
      lambda: diagnose(Q0, "no such section", [LIE0])["fix"] == "both")
check("the diagnosis reports the two inputs, not just the verdict",
      lambda: set(diagnose(Q0, WANT0, [TRUE0])) == {"box", "fix", "evidence", "faithful"})
check("AN END-TO-END SCORE CANNOT SEPARATE THESE TWO RUNS",
      lambda: diagnose(Q0, WANT0, [TRUE0])["box"]
              != diagnose(Q0, missed_section(Q0), [quoted_from_top(Q0)])["box"],
      "both return a fluent, supported-looking answer; only two scores tell them apart")

def _scorecard():
    print(f"  chosen k = {chosen_k()}   recall {recall_at_k(chosen_k()):.0%}   "
          f"precision {precision_at_k(chosen_k()):.0%}\n")
    for q, want, claim in LABELLED:
        d = diagnose(q, want, [claim], k=chosen_k())
        print(f"  {d['box']:28} fix: {d['fix']:12} {q[:40]}")
guard(_scorecard)

## Run it for real

Everything above scored a *supplied* answer. Now let the model write one and score that &mdash; which
is the version you would run against production traces.

In [ ]:
if llm_ready():
    def _score_the_model():
        print(f"  {'box':30}{'fit':>6}{'rel':>6}  question")
        print("  " + "-" * 74)
        for q, want, _ in LABELLED:
            results = search(q, k=chosen_k())
            context = "\n".join(f"- {r['text'][:170]}" for r in results)
            reply = ask(f"Context:\n{context}\n\nQuestion: {q}\n\n"
                        "Answer in one sentence, quoting the context as closely as you can.",
                        system="Be brief and stay inside the context.")
            claims = [reply.strip()]
            d = diagnose(q, want, claims, k=chosen_k())
            print(f"  {d['box']:30}{faithfulness(claims, results):>6.0%}"
                  f"{answer_relevance(q, claims):>6.0%}  {q[:34]}")
    guard(_score_the_model)

### Read it

Expect faithfulness to look poor, and read why before you believe it. This `faithfulness` is an
exact-substring check, so a model that rephrases &mdash; which is what a model does &mdash; scores zero on
a claim that is perfectly well supported. **The metric is measuring quotation, not support.**

That is the honest limitation of a stdlib scorer, and it is the right thing to hit here rather
than in production. Two ways out, and Day 3 uses both:

- ask a model to judge support, and accept that your evaluator is now a model too
- score overlap rather than containment, and pick the threshold with a labelled set

What survives either way is the shape: two numbers, not one, and a 2&times;2 that names which half
to fix.

**What you take from Module 6:** retrieval is a decision, not a step; chunking and metadata decide
more than the embedding does; top-k always returns k, so a floor is what makes refusal possible;
a citation is a span, not a filename; and the two halves are measured apart, or a lucky answer
looks exactly like a good one.

In [ ]:
score()

## Your turn

1. Replace `faithfulness` with a token-overlap score and pick a threshold that accepts a fair
   paraphrase and rejects `LIE0`. How confident are you in that threshold on six examples?
2. Add a seventh labelled question whose answer spans **two** sections. What does recall@k mean
   now, and what did you have to decide about the label?
3. Sweep the floor from Lab 6.4 across 0.1 to 0.5 and plot refusals against correct answers. The
   value you pick is a policy decision about how often you would rather say nothing than be wrong.